## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import gensim
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from myNN import NN

## General functions

In [11]:
language_key = os.getenv("LANGUAGE_KEY")
language_endpoint = os.getenv("LANGUAGE_ENDPOINT")
def authenticate_client():
    ta_credential = AzureKeyCredential(language_key)
    text_analytics_client = TextAnalyticsClient(
            endpoint=language_endpoint,
            credential=ta_credential)
    return text_analytics_client

def load_data(file_path):
    df = pd.read_csv(file_path)
    inputs = df["Text"].values
    outputs = df["Sentiment"].values
    return inputs, outputs

def split_data(inputs, outputs, seedNo = 5, trainSize = 0.8):

    np.random.seed(seedNo)
    noSamples = len(inputs)
    indexes = [i for i in range(noSamples)]

    trainSample = np.random.choice(indexes, size = int(trainSize * noSamples), replace=False)
    testSample = [i for i in indexes if i not in trainSample]

    trainInputs = [inputs[i] for i in trainSample]
    trainOutputs = [outputs[i] for i in trainSample]
    testInputs = [inputs[i] for i in testSample]
    testOutputs = [outputs[i] for i in testSample]

    return trainInputs, trainOutputs, testInputs, testOutputs

def accuracy_score_azure(client, test_in, test_out):

    batch_size = 10
    y_pred = []
    y_true = []

    for i in range(0, len(test_in), batch_size):

        batch_texts = test_in[i:i+batch_size]
        batch_true = test_out[i:i+batch_size]

        result = client.analyze_sentiment(batch_texts, show_opinion_mining=True)

        for doc, true_label in zip(result, batch_true):
            if doc.is_error:
                continue

            y_pred.append(doc.sentiment.lower().strip())
            y_true.append(true_label.lower().strip())

    correct = sum(p == t for p, t in zip(y_pred, y_true))

    return correct / len(y_true)




## General Constants

In [3]:
text = [
    "By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement..",
]

mixed_reviews_file = os.path.join(os.getcwd(), 'data', 'mixed_reviews.csv')
imdb_dataset_file = os.path.join(os.getcwd(), 'data', 'IMDB Dataset.csv')
client = authenticate_client()
inputs, outputs = load_data(mixed_reviews_file)
train_in, train_out, test_in, test_out = split_data(inputs, outputs)
modelPath = os.path.join(os.getcwd(), 'models', 'GoogleNews-vectors-negative300.bin')
word2VecModel300 = gensim.models.KeyedVectors.load_word2vec_format(modelPath, binary=True)



## Azure Client Sentiment - 50p

In [4]:
def sentiment_analysis_azure_client(client, documents):

    result = client.analyze_sentiment(documents, show_opinion_mining=True)
    docs = [doc for doc in result if not doc.is_error]

    for i, doc in enumerate(docs):
        print(f"Text: {documents[i]}")
        print(f"Sentiment: {doc.sentiment}")
        print(f"Confidence scores: {doc.confidence_scores}")
        print()

sentiment_analysis_azure_client(client, documents = text)

Text: By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement..
Sentiment: positive
Confidence scores: {'positive': 0.89, 'neutral': 0.11, 'negative': 0.0}



## Extract characteristics - Bag of Words, TF-IDF, Word2Vec - 50p + other characteristics - 100p

In [5]:
def extract_characteristics_bag_of_words(train_inputs, test_inputs):
    vectorizer = CountVectorizer()
    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)
    print("vocab size: ", len(vectorizer.vocabulary_),  " words")
    print("traindata size: ", len(train_inputs), " emails")
    print("train_features shape: ", train_features.shape)
    print('some words of the vocab: ', vectorizer.get_feature_names_out()[:20])
    print('some features: ', train_features.toarray()[:3])
    print("=======================================================")
    print("testdata size: ", len(test_inputs), " emails")
    print("test_features shape: ", test_features.shape)
    print('some features: ', test_features.toarray()[:3])
    print("=======================================================")
    return train_features, test_features

def extract_characteristics_tfidf(train_inputs, test_inputs):

    vectorizer = TfidfVectorizer(max_features=2000)
    train_features = vectorizer.fit_transform(train_inputs)
    test_features = vectorizer.transform(test_inputs)

    print('vocab train tfidf: ', vectorizer.get_feature_names_out()[:10])
    print('features train tfidf: ', train_features.toarray()[:3])

    return train_features, test_features, vectorizer

def extract_characteristics_word2vec(model, data):
    features = []
    phrases = [ phrase.split() for phrase in data]
    for phrase in phrases:

        vectors = [model[word] for word in phrase if (len(word) > 2) and (word in model.index_to_key)]
        if len(vectors) == 0:
            result = [0.0] * model.vector_size
        else:
            result = np.mean(vectors, axis = 0)
        features.append(result)
    print("features shape(w2v): ", np.array(features).shape)
    return features


def extract_other_features(data):
    features = []
    for text in data:
        words = text.split()
        num_words = len(words)
        num_chars = len(text)
        avg_word_len = num_chars / num_words if num_words > 0 else 0
        num_exclam = text.count("!")
        num_question = text.count("?")
        max_word_len = max((len(w) for w in words), default=0)
        num_caps = sum(1 for w in words if w.isupper())
        features.append([
            num_words,
            num_chars,
            num_caps,
            avg_word_len,
            max_word_len,
            num_exclam,
            num_question
        ])
    print("==================OTHER FEATURES================:")
    print(features[:2])
    return features



In [6]:
#BoW
train_features, test_features = extract_characteristics_bag_of_words(train_in, test_in)

#TF-IDF
train_features_tfidf, test_features_tfidf, vectorizer_tfidf = extract_characteristics_tfidf(train_in, test_in)
text_features_tfidf = vectorizer_tfidf.transform(text)

#W2V
# train_features_w2v = extract_characteristics_word2vec(word2VecModel300, train_in)
# test_features_w2v = extract_characteristics_word2vec(word2VecModel300, test_in)

#Other features
train_other_features = extract_other_features(train_in)
test_other_features = extract_other_features(test_in)

#Data for ANN
X_train = train_features_tfidf.toarray()
X_test = test_features_tfidf.toarray()
X_text = text_features_tfidf.toarray()


vocab size:  595  words
traindata size:  165  emails
train_features shape:  (165, 595)
some words of the vocab:  ['12am' '15th' '1990' '30' '302' '40' '4th' '650' 'about' 'above'
 'abundant' 'ac' 'access' 'across' 'actually' 'added' 'adjust'
 'advertised' 'advised' 'after']
some features:  [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
testdata size:  42  emails
test_features shape:  (42, 595)
some features:  [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
vocab train tfidf:  ['12am' '15th' '1990' '30' '302' '40' '4th' '650' 'about' 'above']
features train tfidf:  [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
==================OTHER FEATURES================:
[[4, 20, 0, 5.0, 6, 0, 0], [11, 53, 0, 4.818181818181818, 8, 0, 0]]
==================OTHER FEATURES================:
[[9, 54, 0, 6.0, 11, 0, 0], [3, 26, 0, 8.666666666666666, 12, 0, 0]]


## ANN cu tool - 100p

In [7]:
mlp = MLPClassifier(
    hidden_layer_sizes=(10,),
    activation='relu',
    max_iter=3000,
    random_state=45
)

mlp.fit(X_train, train_out)
predictions = mlp.predict(X_test)
accuracy_sklearn = accuracy_score(test_out, predictions)
prediction_cerinta = mlp.predict(X_text)
print(f"Accuracy sklearn MLP: {accuracy_sklearn:.2f}")
print("Predictie la textul din cerinta: ", prediction_cerinta)

Accuracy sklearn MLP: 0.83
Predictie la textul din cerinta:  ['negative']


## ANN cu cod propriu - 300p

In [8]:
le = LabelEncoder()
y_train = le.fit_transform(train_out)
y_test = le.transform(test_out)

nn = NN(
    n_features=X_train.shape[1],
    n_classes=len(np.unique(y_train)),
    n_hidden=10
)
nn.fit(X_train, y_train, max_iters=2000, eta=0.1)
pred_test = nn.predict(X_test)
pred_text = nn.predict(X_text)
accuracy_nn = np.mean(pred_test == y_test)
print(f"Accuracy NN manual: {accuracy_nn:.2f}")
print("Predictie la textul din cerinta (NN manual): ", le.inverse_transform(pred_text))

Iteratia 0: loss 0.6934
Iteratia 100: loss 0.6033
Iteratia 200: loss 0.6032
Iteratia 300: loss 0.6030
Iteratia 400: loss 0.6026
Iteratia 500: loss 0.6014
Iteratia 600: loss 0.5976
Iteratia 700: loss 0.5868
Iteratia 800: loss 0.5579
Iteratia 900: loss 0.4936
Iteratia 1000: loss 0.3880
Iteratia 1100: loss 0.2725
Iteratia 1200: loss 0.1855
Iteratia 1300: loss 0.1319
Iteratia 1400: loss 0.1010
Iteratia 1500: loss 0.0830
Iteratia 1600: loss 0.0720
Iteratia 1700: loss 0.0650
Iteratia 1800: loss 0.0603
Iteratia 1900: loss 0.0571
Accuracy NN manual: 0.83
Predictie la textul din cerinta (NN manual):  ['negative']


## Concluzie finala
 - NN este mai bun decat Azure Client deoarece este antrenat pe dataset specific iar azure este mai general

In [12]:
print("Accuracy Azure Client: ", accuracy_score_azure(client, test_in, test_out))
print(f"Accuracy sklearn MLP: {accuracy_sklearn:.2f}")
print(f"Accuracy NN manual: {accuracy_nn:.2f}")

Accuracy Azure Client:  0.6904761904761905
Accuracy sklearn MLP: 0.83
Accuracy NN manual: 0.83
